In [1]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 4090


In [2]:
import os
os.environ["BNB_CUDA_VERSION"] = "124" 
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
from datasets import load_dataset
from transformers import (
AutoTokenizer,
AutoModelForImageTextToText,
BitsAndBytesConfig,
AutoConfig,
EarlyStoppingCallback
)

from peft import (
LoraConfig,
get_peft_model,
prepare_model_for_kbit_training,
TaskType
)

from trl import SFTTrainer, SFTConfig

MODEL_NAME = os.path.realpath("../Qwen3-VL-4B-It")
OUTPUT_DIR = os.path.realpath("../outputs/qwen3_examassist_4b_lora")

/opt/miniconda3/envs/qwen3_stage2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    local_files_only=True,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

In [4]:
dataset = load_dataset(
    "json",
    data_files={
    "train": "../data/stage_1/train-V2.jsonl",
    "validation": "../data/stage_1/valid-V2.jsonl"
    }
)

print(f"Train samples: {len(dataset['train'])}")
print(f"Validation samples: {len(dataset['validation'])}")

# def formatting_func(examples):
#     texts = []
#     # Duyệt qua danh sách các "messages" trong batch dữ liệu
#     for messages in examples["messages"]:
#         text = tokenizer.apply_chat_template(
#             messages, 
#             tokenize=False, 
#             add_generation_prompt=False
#         )
#         texts.append(text)
#     return texts
def formatting_func(example):
    return tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

Train samples: 29595
Validation samples: 3814


In [5]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, 
)

In [6]:
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",   
    local_files_only=True,
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.config.use_cache = False
# model.enable_input_require_grads()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
This can be used to load a bitsandbytes version built with a CUDA version that is different from the PyTorch CUDA version.
If this was unintended set the BNB_CUDA_VERSION variable to an empty string: export BNB_CUDA_VERSION=

Loading weights: 100%|██████████| 713/713 [00:05<00:00, 125.84it/s]


In [7]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ]
)

In [8]:
model = get_peft_model(model, peft_config) 
model.print_trainable_parameters()

trainable params: 33,030,144 || all params: 4,470,845,952 || trainable%: 0.7388


# Training Arguments

In [10]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,

    # Training
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=16,

    # Learning rate
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,

    # Logging
    logging_steps=10,
    logging_first_step=True,

    # Evaluation & Checkpoint
    eval_strategy="steps",
    eval_steps=200,

    save_strategy="steps",
    save_steps=200,

    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Precision
    bf16=True,
    fp16=False,

    # Optimizer
    optim="adamw_torch_fused",

    # Stability
    gradient_checkpointing=True,
    max_grad_norm=0.3,

    # Sequence length
    # max_seq_length=1024,

    # Misc
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=model,

    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],

    args=training_args,

    processing_class=tokenizer,

    formatting_func=formatting_func,
    
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Truncating eval dataset: 100%|██████████| 3814/3814 [00:00<00:00, 333766.10 examples/s]


# ====TRAINING====

In [11]:
trainer.train()

trainer.save_model(OUTPUT_DIR)

tokenizer.save_pretrained(OUTPUT_DIR)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None, 'pad_token_id': 151643}.
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss
200,2.762729,2.818352
400,2.775456,2.818352
600,2.763636,2.818352
800,2.799458,2.818352
1000,2.791926,2.818352
1200,2.783577,2.818352
1389,2.771466,2.818352


('/home/drnguyenvinh/Exam-Assistant/Training_Model/outputs/qwen3_examassist_4b_lora/tokenizer_config.json',
 '/home/drnguyenvinh/Exam-Assistant/Training_Model/outputs/qwen3_examassist_4b_lora/chat_template.jinja',
 '/home/drnguyenvinh/Exam-Assistant/Training_Model/outputs/qwen3_examassist_4b_lora/tokenizer.json')